
# 견적 규격 평가기(Spec Evaluator) 성능 비교: Luna(GPT-5.6) vs Qwen3.5 (RunPod)

`quotation_spec_evaluator.py`의 `LunaQuotationSpecEvaluator`가 지금 쓰고 있는
모델(`gpt-5.6-luna`, OpenAI Responses API)과, 우리가 RunPod에 이미 올려둔
Qwen3.5(+견적 파싱용 LoRA, 구조화 단계는 base 모델)를 **같은 입력·같은 판정 기준**으로
붙여서 비교하기 위한 노트북입니다.

## 실행 전 꼭 확인하세요

1. **RunPod 워커 패치 필요**: 지금 배포된 `handler.py` / `quotation_pipeline.py`는
   구조화 단계 출력을 무조건 견적 추출 스키마(`QuotationExtraction`, `extra="forbid"`)로
   강제 검증합니다. 규격 판정처럼 다른 모양의 JSON을 요청하면 워커가 검증 단계에서
   에러를 냅니다. 이 노트북은 `input.task == "spec_eval"`일 때는 그 강제 검증을
   건너뛰고 모델이 출력한 JSON을 그대로 돌려주도록 패치한 버전을 전제로 합니다.
   (패치본은 대화에서 이미 전달했고, `C:\Users\박동관\Desktop\Qwen3.5\SKNFinal-VLM-Model`에
   반영해뒀습니다. **git commit/push 및 RunPod 재배포는 직접 해주셔야 합니다.**
   재배포 전까지는 아래 RunPod 셀이 계속 에러를 낼 수 있습니다.)
2. **자격 증명**: RunPod 엔드포인트 ID/API 키는 갖고 계신다고 하셨고, OpenAI API 키는
   `.env`에 있는 걸 그대로 쓰시면 됩니다. 아래 "자격 증명" 셀에서 입력하세요
   (노트북 파일 자체에는 값을 남기지 마세요 — 실행 후 저장 전에 지우거나, `getpass`로
   매번 입력하는 방식을 쓰세요).
3. **콜드스타트**: RunPod Active workers=0이면 첫 호출은 모델 로딩(수십 초~수 분)까지
   포함된 시간이 찍힙니다. 워밍업 1회 이후 다시 측정하는 것도 같이 보시는 걸 추천합니다.
4. **MAX_NEW_TOKENS_CAP**: 견적 1건당 규격 판정 JSON이 꽤 길 수 있습니다(품목별 상세
   포함). RunPod 엔드포인트 환경변수의 `MAX_NEW_TOKENS_CAP`이 너무 작으면 출력이
   잘릴 수 있으니, 필요하면 900~1200 정도로 올려두세요.

## 이 노트북이 하는 일

- RFQ 1건(화학물질용 완전밀폐형 전신 보호복, EN 943-1 Type 1a-ET) + 업로드해주신
  합성 데이터셋의 까다로운 견적 10건(자인형 부적합, 경계값 함정, 단위 환산 함정,
  모호/누락형 부적합, 우량 공급사 오판 함정 등)을 텍스트로만 구성합니다
  (문서 파싱은 이미 끝났다고 가정 — 이미지 없이 `document_text`만 사용).
- 견적마다 Luna와 RunPod Qwen3.5에 **동일한 심사 기준(INSTRUCTIONS)** 을 주고
  규격 적합성 판정을 시킨 뒤, 지연시간/구조적 유효성/정답(직접 라벨링한 answer key)
  일치 여부를 비교표로 보여줍니다.
- 실제 전환 여부는 이 결과 + 실제 운영 데이터로 몇 차례 더 돌려본 뒤에 판단하시길
  권장합니다 (이 10건은 스트레스 테스트용 트랩이라, 실제 분포보다 어렵게 만들었고,
  answer key는 제가 직접 채점한 것이라 이견이 있을 수 있습니다).


In [10]:

import hashlib
import json
import os
import time
from dataclasses import dataclass, field
from typing import Any

import requests
from pydantic import BaseModel, ConfigDict, Field

try:
    import pandas as pd
except ImportError:  # pragma: no cover
    pd = None


## 1. 자격 증명

이미 갖고 계신 값을 넣으세요. `.env`에 있는 것과 이름을 맞췄습니다.
값을 직접 셀에 타이핑하고 싶지 않다면 `getpass.getpass()`로 바꿔서 매번 입력하는
방식을 쓰셔도 됩니다.

In [11]:

# 필요하면 getpass로 바꿔서 노트북 파일에 값이 남지 않게 하세요.
# import getpass
# OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
RUNPOD_QUOTATION_ENDPOINT_ID = os.getenv("RUNPOD_QUOTATION_ENDPOINT_ID", "")
RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY", "")

# 환경변수에 없으면 여기 직접 채워넣으세요 (실행 후 지우는 걸 잊지 마세요).
# OPENAI_API_KEY = "sk-..."
# RUNPOD_QUOTATION_ENDPOINT_ID = "xxxxxxxxxxxxxxxx"
# RUNPOD_API_KEY = "rpa_..."

assert OPENAI_API_KEY, "OPENAI_API_KEY를 설정하세요."
assert RUNPOD_QUOTATION_ENDPOINT_ID, "RUNPOD_QUOTATION_ENDPOINT_ID를 설정하세요."
assert RUNPOD_API_KEY, "RUNPOD_API_KEY를 설정하세요."

RUNPOD_API_BASE_URL = "https://api.runpod.ai/v2"


## 2. 판정 스키마 & 심사 기준 (`quotation_spec_evaluator.py`와 동일)

아래 pydantic 모델과 `INSTRUCTIONS` 문구는 실제 운영 코드
(`backend_logic2/nodes/quotation/quotation_filter/quotation_spec_evaluator.py`)에서
그대로 가져온 것입니다. 두 모델이 **완전히 같은 심사 기준**으로 평가받도록, 나중에
운영 코드가 바뀌면 여기도 같이 업데이트해주세요.

In [21]:

class SpecItemAssessment(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    quotation_item: str
    rfq_item: str
    compliant: bool
    score: float = Field(ge=0, le=100)
    reason: str
    missing_or_conflicting_specs: list[str] = Field(default_factory=list)


class QuotationSpecAssessment(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    quotation_id: str
    compliant: bool
    score: float = Field(ge=0, le=100)
    confidence: float = Field(ge=0, le=1)
    reason: str
    items: list[SpecItemAssessment]


class QuotationSpecAssessmentBatch(BaseModel):
    model_config = ConfigDict(extra="forbid")

    assessments: list[QuotationSpecAssessment]


INSTRUCTIONS = """
당신은 산업 구매 견적의 기술 규격 적합성을 판정하는 심사자입니다.
RFQ 요구사항과 각 Supplier Quotation의 품목 설명, 원문 설명, 구조화된 specifications만 비교하세요.
가격, 공급사 인지도, 납기, 과거 실적은 규격 점수에 절대 반영하지 마세요.

판정 원칙:
- 동의어, 약어, 단위 환산, 표기 순서 차이는 의미가 같으면 일치로 봅니다.
- 수치 허용오차가 명시되면 그것을 적용하고, 명시되지 않으면 임의로 허용오차를 만들지 않습니다.
- 상위 규격이 요구 규격을 완전히 포함한다는 기술적 근거가 있을 때만 적합으로 봅니다.
- 필수 규격이 확인되지 않거나 상충하면 compliant=false로 판정하고 무엇이 부족한지 적습니다.
- 추측하지 말고 제공된 근거만 사용합니다.
- score는 모든 필수 규격 충족도를 0~100으로 표현하며, compliant는 필수 규격을 모두 충족할 때만 true입니다.
- 각 quotation_id마다 정확히 하나의 assessment를 반환합니다.

출력 분량 제한 (판정 자체를 바꾸지 말고 서술 분량만 줄이세요):
- 최상위 reason은 1문장으로 핵심만 요약하세요. 품목별 상세 사유를 여기서 반복하지 마세요.
- 품목별(items) reason도 1~2문장으로 제한하세요.
- missing_or_conflicting_specs 항목은 완전한 문장이 아니라 "방폭등급 미달", "재질 SUS304<SUS316" 같은 짧은 구(phrase)로만 쓰세요.
""".strip()


## 3. 테스트 데이터: RFQ 1건 + 복잡한 견적 10건 (업로드해주신 합성 데이터셋)

전제는 동일합니다 — 포털/엑셀/PDF/이미지 파싱은 이미 끝났다고 가정하고, 파싱 결과에
해당하는 텍스트(구조화된 `specifications` + `raw_description`/`notes` 자유서술)만 씁니다.
이미지는 전혀 안 씁니다.

RFQ 품목은 **화학물질용 완전밀폐형 전신 보호복(EN 943-1 Type 1a-ET, Level A급)** 1종이고,
요구 규격이 13개(보호복 형식/인증/재질/투과저항/봉제부/안면창/장갑/안전화/기밀시험/
사용온도/공기호흡기/보관수명/사이즈)나 되는 훨씬 복잡한 시나리오입니다. 견적 10건에는
제가 처음에 만들었던 6건보다 실제로 더 어려운 트랩들이 들어있습니다:

- **자인형 부적합** (Q3, Q4, Q8): 공급사가 notes에 스스로 "요구사항과 다르다"고 적어둔 케이스.
  구조화 필드만 보면 얼핏 비슷해 보일 수 있어 이 자백을 놓치지 않는지가 관건.
- **경계값 함정** (Q5, Q10): 구조화 스펙 자체는 요구치를 "정확히" 충족(0.50mm, 480분, 80도,
  200J, 5년 등 — 이상/이내의 경계값)하는데, Q5는 필수 인증(KCs)이 notes에서 "발급 보장 불가"로
  흔들리고, Q10은 notes에 "재고 상황에 따라 완전히 다른(인증 없는) 대체모델로 바뀔 수 있다"는
  리스크가 숨어있음. 구조화 필드만 보고 안이하게 통과시키면 안 되는 케이스들.
- **단위 환산 함정** (Q6, Q9): 필수 스펙 값들이 마이크로미터/시간/켈빈/라디안/개월 등
  RFQ와 다른 단위로 적혀있어서, mm/분/섭씨/도/년으로 직접 환산해야 충족 여부를 판단할 수 있음.
  Q9는 여기에 전체 영어 표기까지 겹쳐서 이번 세트에서 가장 어려운 케이스입니다.
- **모호/누락형 부적합** (Q7): 규격이 "고내화학 복합 원단", "해외 인증 제품"처럼 뭉뚱그려져
  있어서 사실상 확인이 불가능한 케이스.
- **우량 공급사 오판 함정** (Q2): 전부 요구치보다 우수한데, 표기가 달라서(EN943-2 추가 인증,
  삼중 열융착 등) 오히려 미달로 오판하기 쉬운 케이스.

`ANSWER_KEY`는 제가 위 기준으로 직접 채점한 정답이며(compliant=True 4건 / False 6건),
**모델 입력에는 포함하지 않습니다.** 다만 이건 사람이 판단한 것이라 이견이 있을 수 있는
지점(특히 Q2, Q5, Q10)도 있으니, 결과 보시고 이상하다 싶으면 말씀해주세요 — 정답 자체를
같이 다시 논의해도 됩니다.

In [29]:
RFQ = {'rfq_name': 'PUR-RFQ-SPEC-TEST-001',
 'currency': 'KRW',
 'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
            'item_name': '화학물질용 완전밀폐형 전신 보호복',
            'quantity': 20,
            'required_delivery_date': '2026-11-30',
            'numeric_tolerance_percent': 0,
            'specifications': {'보호복 형식': 'EN 943-1 Type 1a-ET, 완전 밀폐형, 보호복 내부에 SCBA 착용',
                               '필수 인증': 'KCs 안전인증 및 CE Category III',
                               '보호복 재질': '다층 EVOH 배리어 원단, 전체 두께 0.50 mm 이상',
                               '화학물질 투과 저항': '황산 98% 480분 이상, 암모니아수 25% 240분 이상, 염소가스 60분 이상',
                               '봉제부': '이중 열융착 및 배리어 테이프 마감, 보호복 본체와 동등한 투과 저항',
                               '안면창': '김서림 방지, 수평 시야각 80도 이상, 현장 교체 가능',
                               '보호 장갑': '교체형 이중 장갑, 내부 Butyl 및 외부 Viton 재질',
                               '안전화': '일체형 EN ISO 20345 S5 SRC 안전화, 토캡 충격 저항 200 J 이상',
                               '기밀 시험': '제품별 EN 464 기밀 시험 및 일련번호별 시험성적서 제출',
                               '사용 온도': '-30℃~+60℃',
                               '공기호흡기': '보호복 내부에 6 L~9 L SCBA 착용 가능',
                               '보관 수명': '제조일 기준 5년 이상',
                               '공급 사이즈': 'M, L, XL 공급 가능'}}]}

# 각 quotation 은 quotation_spec_evaluator.evaluate() 가 실제로 만드는 payload와
# 동일한 모양(quotation_id/supplier_name/items[item_code,item_name,description,
# raw_description,specifications]/notes)으로 맞췄습니다.
# 이 10건은 실제 업로드해주신 합성 테스트셋(synthetic_erpnext_complex_chemical_suit_quotations_10)을
# 그대로 쓰고, _case_id/_trap만 제가 채점 편의를 위해 붙였습니다.

QUOTATIONS = [{'_case_id': 'Q1_가상공급사A_완전충족',
  '_trap': '전 항목이 요구치를 그대로 충족하거나 상회하는 깨끗한 기준선.',
  'quotation_id': 'SUP-QTN-SPEC-001',
  'supplier_name': '가상공급사-A',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'AlphaShield 완전밀폐형 화학보호복',
             'description': 'EN 943-1 Type 1a-ET 완전 밀폐형 보호복. 내부 SCBA 방식. KCs 및 CE Category III 인증 '
                            '제품.',
             'raw_description': 'AlphaShield Type 1a-ET / KCs·CE III / EVOH 0.52 mm / 내부 SCBA 6~9 '
                                'L / EN 464 전수검사',
             'specifications': {'type': 'EN 943-1 Type 1a-ET, gas-tight, internal SCBA',
                                'certification': 'KCs 안전인증, CE Category III',
                                'material': '다층 EVOH 배리어, 두께 0.52 mm',
                                'permeation': '황산 98% 510분; 암모니아수 25% 260분; 염소가스 75분',
                                'seam': '이중 열융착 및 배리어 테이프, 본체 동등 등급',
                                'visor': '김서림 방지, 수평 시야각 84도, 현장 교체형',
                                'gloves': '교체형 Butyl 내피 및 Viton 외피 이중 장갑',
                                'boots': 'EN ISO 20345 S5 SRC 일체형, 토캡 200 J',
                                'leak_test': '전 제품 EN 464 시험, 일련번호별 성적서 제공',
                                'temperature': '-35℃~+65℃',
                                'scba': '내부 6 L, 6.8 L, 9 L 지원',
                                'shelf_life': '제조일 기준 6년',
                                'sizes': 'M, L, XL'}}],
  'notes': '납품 시 제품별 EN 464 시험성적서와 인증서 사본을 함께 제출합니다.'},
 {'_case_id': 'Q2_가상공급사B_상위규격_우량공급사',
  '_trap': "전 항목이 요구치보다 명확히 우수(EVOH 7겹, 투과저항 전부 초과, 온도범위 더 넓음 등). 'EN943-2 추가 인증', '삼중 열융착'처럼 "
           '요구사항과 표기가 달라 오히려 미달로 오판하기 쉬운 표현이 섞여있음.',
  'quotation_id': 'SUP-QTN-SPEC-002',
  'supplier_name': '가상공급사-B',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'ChemGuard Pro 1a-ET',
             'description': 'EN 943-1 Type 1a-ET 및 EN 943-2 비상대응팀용 요구사항을 충족하는 완전 밀폐형 보호복.',
             'raw_description': 'ChemGuard Pro, EVOH 7-layer 0.58 mm, Type 1a-ET, KCs + CE III, '
                                'internal SCBA',
             'specifications': {'classification': 'EN 943-1 Type 1a-ET 및 EN 943-2',
                                'approvals': 'KCs 안전인증, CE PPE Category III',
                                'fabric': 'EVOH 복합 7층 배리어 원단, 0.58 mm',
                                'breakthrough': '98% H2SO4 600분; 25% NH4OH 360분; Cl2 90분',
                                'seams': '삼중 열융착 후 배리어 스트립 마감',
                                'window': '안티포그, 수평 시야각 88도, 무공구 교체',
                                'hand_protection': 'Butyl/Viton 교체형 이중 장갑',
                                'footwear': 'EN ISO 20345 S5 SRC, 200 J 일체형 안전화',
                                'testing': 'EN 464 전수 기밀시험 및 QR 일련번호 성적서',
                                'operating_temperature': '-40℃~+70℃',
                                'breathing_apparatus': '내부형 SCBA 6~9 L',
                                'storage_life': '7년',
                                'size_range': 'M/L/XL/2XL'}}],
  'notes': 'RFQ 요구 규격보다 높은 화학물질 투과 저항과 사용 온도 범위를 보증합니다.'},
 {'_case_id': 'Q3_가상공급사C_외부SCBA_자인',
  '_trap': "Type 1b-ET(외부 공기호흡기)로 RFQ가 요구하는 Type 1a-ET 내부 SCBA 방식과 다름. notes에서 공급사 스스로 'RFQ의 내부 "
           "SCBA 방식과 다릅니다'라고 인정함 — 가장 명확한 부적합 케이스.",
  'quotation_id': 'SUP-QTN-SPEC-003',
  'supplier_name': '가상공급사-C',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'Responder Type 1b-ET 보호복',
             'description': 'EN 943-1 Type 1b-ET 외부형 SCBA 완전 밀폐 보호복.',
             'raw_description': 'Type 1b-ET external SCBA configuration; EVOH 0.55 mm',
             'specifications': {'type': 'EN 943-1 Type 1b-ET, 외부 SCBA',
                                'certification': 'KCs, CE Category III',
                                'material': 'EVOH 다층 원단 0.55 mm',
                                'permeation': '황산 98% 500분; 암모니아수 25% 250분; 염소가스 65분',
                                'seam': '이중 열융착 및 배리어 테이프',
                                'visor': '안티포그, 시야각 82도, 교체 가능',
                                'gloves': 'Butyl/Viton 이중 장갑',
                                'boots': 'S5 SRC, 200 J',
                                'leak_test': 'EN 464 개별 시험',
                                'temperature': '-30℃~+60℃',
                                'scba': '외부 6 L~9 L',
                                'shelf_life': '5년',
                                'sizes': 'M/L/XL'}}],
  'notes': '외부 공기호흡기 방식이므로 실린더 교체는 빠르지만 RFQ의 내부 SCBA 방식과 다릅니다.'},
 {'_case_id': 'Q4_가상공급사D_전면미달_자인',
  '_trap': '재질(PVC코팅, 0.45mm), 투과저항(전부 요구치 미달), 시야각(75도), 장갑(단일 고정형), 안전화(100J), 온도범위, 보관수명(3년), '
           '사이즈(M 누락) 등 거의 전 항목 미달. notes에서 KCs 인증과 EN464 개별 성적서 미제공까지 자인함 — 최저가지만 총체적 부적합.',
  'quotation_id': 'SUP-QTN-SPEC-004',
  'supplier_name': '가상공급사-D',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'SafeFlex 밀폐형 화학보호복',
             'description': 'Type 1a 계열 내부 SCBA 보호복. PVC 코팅 원단 적용.',
             'raw_description': 'PVC coated Type 1a suit, internal 6 L SCBA, CE III',
             'specifications': {'type': 'EN 943-1 Type 1a, 내부 SCBA',
                                'certification': 'CE Category III',
                                'material': 'PVC 코팅 폴리에스터 0.45 mm',
                                'permeation': '황산 98% 180분; 암모니아수 25% 120분; 염소가스 자료 없음',
                                'seam': '봉제 후 단면 테이프 마감',
                                'visor': '김서림 방지, 시야각 75도, 공장 교체',
                                'gloves': '고정형 Butyl 단일 장갑',
                                'boots': 'SRC 일체형 안전화, 토캡 100 J',
                                'leak_test': '출고 로트별 기밀시험',
                                'temperature': '-10℃~+50℃',
                                'scba': '내부 6 L',
                                'shelf_life': '3년',
                                'sizes': 'L/XL'}}],
  'notes': 'KCs 인증과 EN 464 개별 시험성적서는 제공되지 않습니다.'},
 {'_case_id': 'Q5_가상공급사E_KCs인증_미확정',
  '_trap': "구조화 스펙은 전부 요구치를 정확히 '경계값'으로 충족(0.50mm, 480/240/60분, 80도, 200J, 5년 등 — 이상/이내 경계값 자체를 "
           "미달로 착각하면 안 되는 케이스). 다만 notes에서 필수 인증인 KCs가 '갱신 심사 중이며 납품일까지 발급을 보장할 수 없다'고 명시 — 이 한 가지 "
           '때문에 부적합.',
  'quotation_id': 'SUP-QTN-SPEC-005',
  'supplier_name': '가상공급사-E',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'BarrierMax ET 전신 보호복',
             'description': 'EN 943-1 1a-ET 내부형 공기호흡기 보호복. 인증 갱신 심사 중.',
             'raw_description': 'BarrierMax ET, Type 1a-ET, EVOH 0.50 mm, internal SCBA',
             'specifications': {'type': 'EN 943-1 Type 1a-ET, internal SCBA',
                                'material': 'EVOH 6층 배리어 0.50 mm',
                                'permeation': '황산 98% 480분; 암모니아수 25% 240분; 염소가스 60분',
                                'seam': '이중 열융착 및 양면 배리어 테이프',
                                'visor': '안티포그, 시야각 80도, 교체형',
                                'gloves': '교체형 Butyl/Viton 이중 장갑',
                                'boots': 'EN ISO 20345 S5 SRC, 200 J',
                                'leak_test': 'EN 464 전수 시험 및 개별 성적서',
                                'temperature': '-30℃~+60℃',
                                'scba': '내부 6~9 L',
                                'shelf_life': '5년',
                                'sizes': 'M/L/XL'}}],
  'notes': 'CE Category III 인증서는 제출 가능합니다. 국내 KCs 인증은 갱신 심사 중이며 납품일까지 발급을 보장할 수 없습니다.'},
 {'_case_id': 'Q6_가상공급사F_구조화누락_단위환산',
  '_trap': 'specifications 필드엔 13개 요구 규격 중 4개만 있고 나머지(투과저항/봉제부/안면창/장갑/안전화/기밀시험/공기호흡기/보관수명)는 전부 '
           'notes 자유서술에만 있음. 게다가 재질은 마이크로미터(500 μm), 온도는 켈빈(243.15K~333.15K), 투과저항은 시간 '
           '단위(8시간/4시간/1시간)로 되어있어 mm/섭씨/분 단위로 직접 환산해야 요구치 충족 여부를 알 수 있음. 다 맞으면 전부 충족.',
  'quotation_id': 'SUP-QTN-SPEC-006',
  'supplier_name': '가상공급사-F',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'HazMat One 밀폐형 보호복',
             'description': 'Type 1a-ET, KCs 및 CE III, EVOH 배리어 500 μm, 내부 SCBA 방식.',
             'raw_description': 'HazMat One / Type 1a-ET / KCs / CE Cat III / EVOH 500 microns',
             'specifications': {'type': 'EN 943-1 Type 1a-ET',
                                'certification': 'KCs; CE Category III',
                                'material': 'multilayer EVOH 500 μm',
                                'temperature': '243.15 K~333.15 K',
                                'sizes': 'M, L, XL'}}],
  'notes': '[상세 사양] 황산 98% 8시간, 암모니아수 25% 4시간, 염소가스 1시간 투과 저항. 이중 열융착과 배리어 테이프 마감. 안티포그 교체형 안면창 수평 '
           '시야각 82도. 교체형 Butyl 내피/Viton 외피 장갑. EN ISO 20345 S5 SRC 200 J 일체형 안전화. 전 제품 EN 464 '
           '시험성적서 제공. 내부 6~9 L SCBA 사용. 보관 수명 5년.'},
 {'_case_id': 'Q7_가상공급사G_스펙모호_사이즈누락',
  '_trap': "규격 대부분이 '고내화학 복합 원단', '해외 인증 제품'처럼 모호하게만 기재되어 EN943-1 Type 1a-ET, EVOH, KCs 인증 여부를 전혀 "
           "확인할 수 없음. notes에도 '세부 시험성적과 투과 저항은 발주 후 제공 예정'이라 사실상 검증 불가. 사이즈도 M이 빠져있음 — 확인 불가/누락 "
           '사유로 부적합.',
  'quotation_id': 'SUP-QTN-SPEC-007',
  'supplier_name': '가상공급사-G',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': '산업용 화학보호복',
             'description': '완전 밀폐형 화학보호복, 공기호흡기 사용 가능.',
             'raw_description': '산업용 완전 밀폐 화학보호복, 복합 원단, 해외 인증',
             'specifications': {'type': '완전 밀폐형',
                                'material': '고내화학 복합 원단',
                                'certification': '해외 인증 제품',
                                'sizes': 'L/XL'}}],
  'notes': '세부 시험성적과 투과 저항은 발주 후 제조사 자료로 제공 예정입니다.'},
 {'_case_id': 'Q8_가상공급사H_다른표준_자인',
  '_trap': 'EN 14605 Type 3(액밀형/비밀폐형)로, RFQ가 요구하는 EN 943-1 Type 1a-ET(가스 기밀 완전밀폐형)와 근본적으로 다른 등급의 '
           "제품. notes에서 '가스 기밀형 Type 1 보호복이 아니며 EN464 시험 대상이 아님'이라고 공급사가 직접 인정 — Q3와는 다른 종류의(하위 호환 "
           '아닌 아예 다른 표준) 자인 케이스.',
  'quotation_id': 'SUP-QTN-SPEC-008',
  'supplier_name': '가상공급사-H',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'LiquidSplash Type 3 보호복',
             'description': 'EN 14605 Type 3 액밀형 화학보호복. 비밀폐형 후드와 외부 호흡기 사용.',
             'raw_description': 'EN 14605 Type 3 liquid-tight splash suit, non-gas-tight',
             'specifications': {'type': 'EN 14605 Type 3 liquid-tight, 비밀폐형',
                                'certification': 'CE Category III',
                                'material': 'PVC/폴리에스터 0.42 mm',
                                'permeation': '황산 50% 120분',
                                'seam': '봉제 및 단면 테이프',
                                'visor': '고정형 후드 안면창',
                                'gloves': 'Nitrile 단일 장갑',
                                'boots': '교체형 고무 장화',
                                'temperature': '0℃~+45℃',
                                'sizes': 'M/L/XL'}}],
  'notes': '본 제품은 가스 기밀형 Type 1 보호복이 아니며 EN 464 시험 대상이 아닙니다.'},
 {'_case_id': 'Q9_가상공급사I_영문_다중단위환산',
  '_trap': '설명/규격이 전부 영어로 되어있고, 마이크로미터(500 micrometres), 시간(8h/4h/1h), 켈빈(243.15K~333.15K), '
           '라디안(1.43 rad ≈ 82도), 개월(60 months=5년)까지 다섯 가지 단위를 전부 정확히 환산해야 요구치 충족 여부를 알 수 있는, 이번 '
           '세트에서 가장 어려운 케이스. 다 맞으면 전부 정확히 최소기준을 충족함.',
  'quotation_id': 'SUP-QTN-SPEC-009',
  'supplier_name': '가상공급사-I',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'GlobalBarrier A-ET Suit',
             'description': 'Gas-tight emergency-team suit with breathing apparatus worn inside '
                            'the suit.',
             'raw_description': 'EN943-1 1a-ET / EVOH 500 μm / 8 h H2SO4 / 4 h NH4OH / 1 h Cl2 / '
                                'internal SCBA',
             'specifications': {'classification': 'EN943-1 1a-ET; fully encapsulating',
                                'approvals': 'Korean KCs safety mark and CE PPE Cat. III',
                                'fabric': 'seven-layer EVOH laminate, 500 micrometres',
                                'breakthrough_time': '98% sulfuric acid 8 h; 25% ammonium '
                                                     'hydroxide 4 h; chlorine 1 h',
                                'seam_construction': 'double welded and barrier taped, suit-body '
                                                     'equivalent',
                                'face_lens': 'anti-mist, 1.43 rad horizontal field, field '
                                             'replaceable',
                                'glove_system': 'replaceable Butyl inner and Viton outer gloves',
                                'foot_protection': 'EN ISO 20345 S5 SRC, 200 joule toe cap',
                                'quality_test': '100% EN464 leak test with serialized report',
                                'service_temperature': '243.15 K to 333.15 K',
                                'scba_capacity': 'internal 6, 6.8 or 9 litre cylinder',
                                'storage_period': '60 months from manufacture',
                                'available_sizes': 'M, L and XL'}}],
  'notes': 'All specification values are stated in equivalent SI units. Individual test reports '
           'and certificates are included.'},
 {'_case_id': 'Q10_가상공급사J_구조화vs원문_대체모델리스크',
  '_trap': "specifications 필드만 보면 Q1/Q5처럼 완벽하게 요구치를 충족하는 것처럼 보임. 하지만 notes에 '재고 상황에 따라 Type 1b-ET "
           "외부 SCBA형으로 대체될 수 있고, 대체 모델은 KCs 인증이 없다'는 조건부 리스크가 숨어있음 — 구조화 필드만 보고 통과시키면 안 되는, Q6(밸브 "
           "테스트)의 '구조화-원문 상충' 트랩을 더 정교하게 만든 버전.",
  'quotation_id': 'SUP-QTN-SPEC-010',
  'supplier_name': '가상공급사-J',
  'items': [{'item_code': 'PPE-CHEM-LEVEL-A',
             'item_name': 'DualSpec Emergency Suit',
             'description': '제안 규격표 기준 EN 943-1 Type 1a-ET 내부 SCBA 보호복.',
             'raw_description': 'DualSpec Type 1a-ET internal SCBA, EVOH 0.50 mm',
             'specifications': {'type': 'EN 943-1 Type 1a-ET, 내부 SCBA',
                                'certification': 'KCs 및 CE Category III',
                                'material': 'EVOH 다층 배리어 0.50 mm',
                                'permeation': '황산 98% 480분; 암모니아수 25% 240분; 염소가스 60분',
                                'seam': '이중 열융착 및 배리어 테이프',
                                'visor': '안티포그, 시야각 80도, 교체 가능',
                                'gloves': 'Butyl/Viton 교체형 이중 장갑',
                                'boots': 'S5 SRC, 200 J',
                                'leak_test': 'EN 464 전수시험 및 일련번호 성적서',
                                'temperature': '-30℃~+60℃',
                                'scba': '내부 6~9 L',
                                'shelf_life': '5년',
                                'sizes': 'M/L/XL'}}],
  'notes': '실제 납품 모델은 재고 상황에 따라 EN 943-1 Type 1b-ET 외부 SCBA형으로 대체될 수 있습니다. 대체 모델은 KCs 인증이 없고 CE '
           'Category III만 보유합니다.'}]

ANSWER_KEY = {'SUP-QTN-SPEC-001': {'expected_compliant': True, 'note': '전 항목이 요구치를 그대로 충족하거나 상회하는 깨끗한 기준선.'},
 'SUP-QTN-SPEC-002': {'expected_compliant': True,
                      'note': "전 항목이 요구치보다 명확히 우수(EVOH 7겹, 투과저항 전부 초과, 온도범위 더 넓음 등). 'EN943-2 추가 "
                              "인증', '삼중 열융착'처럼 요구사항과 표기가 달라 오히려 미달로 오판하기 쉬운 표현이 섞여있음."},
 'SUP-QTN-SPEC-003': {'expected_compliant': False,
                      'note': 'Type 1b-ET(외부 공기호흡기)로 RFQ가 요구하는 Type 1a-ET 내부 SCBA 방식과 다름. notes에서 '
                              "공급사 스스로 'RFQ의 내부 SCBA 방식과 다릅니다'라고 인정함 — 가장 명확한 부적합 케이스."},
 'SUP-QTN-SPEC-004': {'expected_compliant': False,
                      'note': '재질(PVC코팅, 0.45mm), 투과저항(전부 요구치 미달), 시야각(75도), 장갑(단일 고정형), '
                              '안전화(100J), 온도범위, 보관수명(3년), 사이즈(M 누락) 등 거의 전 항목 미달. notes에서 KCs 인증과 '
                              'EN464 개별 성적서 미제공까지 자인함 — 최저가지만 총체적 부적합.'},
 'SUP-QTN-SPEC-005': {'expected_compliant': False,
                      'note': "구조화 스펙은 전부 요구치를 정확히 '경계값'으로 충족(0.50mm, 480/240/60분, 80도, 200J, 5년 등 "
                              "— 이상/이내 경계값 자체를 미달로 착각하면 안 되는 케이스). 다만 notes에서 필수 인증인 KCs가 '갱신 심사 "
                              "중이며 납품일까지 발급을 보장할 수 없다'고 명시 — 이 한 가지 때문에 부적합."},
 'SUP-QTN-SPEC-006': {'expected_compliant': True,
                      'note': 'specifications 필드엔 13개 요구 규격 중 4개만 있고 '
                              '나머지(투과저항/봉제부/안면창/장갑/안전화/기밀시험/공기호흡기/보관수명)는 전부 notes 자유서술에만 있음. 게다가 '
                              '재질은 마이크로미터(500 μm), 온도는 켈빈(243.15K~333.15K), 투과저항은 시간 '
                              '단위(8시간/4시간/1시간)로 되어있어 mm/섭씨/분 단위로 직접 환산해야 요구치 충족 여부를 알 수 있음. 다 맞으면 '
                              '전부 충족.'},
 'SUP-QTN-SPEC-007': {'expected_compliant': False,
                      'note': "규격 대부분이 '고내화학 복합 원단', '해외 인증 제품'처럼 모호하게만 기재되어 EN943-1 Type 1a-ET, "
                              "EVOH, KCs 인증 여부를 전혀 확인할 수 없음. notes에도 '세부 시험성적과 투과 저항은 발주 후 제공 "
                              "예정'이라 사실상 검증 불가. 사이즈도 M이 빠져있음 — 확인 불가/누락 사유로 부적합."},
 'SUP-QTN-SPEC-008': {'expected_compliant': False,
                      'note': 'EN 14605 Type 3(액밀형/비밀폐형)로, RFQ가 요구하는 EN 943-1 Type 1a-ET(가스 기밀 '
                              "완전밀폐형)와 근본적으로 다른 등급의 제품. notes에서 '가스 기밀형 Type 1 보호복이 아니며 EN464 시험 "
                              "대상이 아님'이라고 공급사가 직접 인정 — Q3와는 다른 종류의(하위 호환 아닌 아예 다른 표준) 자인 케이스."},
 'SUP-QTN-SPEC-009': {'expected_compliant': True,
                      'note': '설명/규격이 전부 영어로 되어있고, 마이크로미터(500 micrometres), 시간(8h/4h/1h), '
                              '켈빈(243.15K~333.15K), 라디안(1.43 rad ≈ 82도), 개월(60 months=5년)까지 다섯 가지 '
                              '단위를 전부 정확히 환산해야 요구치 충족 여부를 알 수 있는, 이번 세트에서 가장 어려운 케이스. 다 맞으면 전부 정확히 '
                              '최소기준을 충족함.'},
 'SUP-QTN-SPEC-010': {'expected_compliant': False,
                      'note': "specifications 필드만 보면 Q1/Q5처럼 완벽하게 요구치를 충족하는 것처럼 보임. 하지만 notes에 '재고 "
                              "상황에 따라 Type 1b-ET 외부 SCBA형으로 대체될 수 있고, 대체 모델은 KCs 인증이 없다'는 조건부 리스크가 "
                              "숨어있음 — 구조화 필드만 보고 통과시키면 안 되는, Q6(밸브 테스트)의 '구조화-원문 상충' 트랩을 더 정교하게 만든 "
                              '버전.'}}

print(f"RFQ 품목 {len(RFQ['items'])}개, 테스트 견적 {len(QUOTATIONS)}건 준비 완료")
print(f"정답 분포: compliant=True {sum(1 for v in ANSWER_KEY.values() if v['expected_compliant'])}건 / "
      f"False {sum(1 for v in ANSWER_KEY.values() if not v['expected_compliant'])}건")

RFQ 품목 1개, 테스트 견적 10건 준비 완료
정답 분포: compliant=True 4건 / False 6건


## 4. Luna (OpenAI `gpt-5.6-luna`) 호출

운영 코드의 `LunaQuotationSpecEvaluator.evaluate()`와 동일한 방식(`responses.parse` +
`text_format=QuotationSpecAssessmentBatch`)으로, **견적 1건씩** 호출합니다
(6건을 한 번에 묶으면 RunPod 쪽과 배치 크기가 안 맞아 지연시간 비교가 왜곡됩니다).

In [14]:

from openai import OpenAI

_openai_client = OpenAI(api_key=OPENAI_API_KEY, timeout=60)

LUNA_MODEL = os.getenv("QUOTATION_SPEC_MODEL", "gpt-5.6-luna")
LUNA_REASONING_EFFORT = os.getenv("QUOTATION_SPEC_REASONING_EFFORT", "medium")
LUNA_MAX_OUTPUT_TOKENS = int(os.getenv("QUOTATION_SPEC_MAX_OUTPUT_TOKENS", "4000"))


def call_luna(rfq: dict, quotation: dict) -> dict:
    """단일 견적에 대해 Luna로 규격 평가를 1회 호출한다."""

    payload = {
        "rfq": rfq,
        "quotations": [
            {
                "quotation_id": quotation["quotation_id"],
                "supplier_name": quotation["supplier_name"],
                "items": [
                    {
                        "item_code": item["item_code"],
                        "item_name": item["item_name"],
                        "description": item["description"],
                        "raw_description": item["raw_description"],
                        "specifications": item["specifications"],
                    }
                    for item in quotation["items"]
                ],
                "notes": quotation["notes"],
            }
        ],
    }

    started = time.perf_counter()
    error = None
    parsed_batch = None
    try:
        response = _openai_client.responses.parse(
            model=LUNA_MODEL,
            instructions=INSTRUCTIONS,
            input=json.dumps(payload, ensure_ascii=False),
            reasoning={"effort": LUNA_REASONING_EFFORT},
            max_output_tokens=LUNA_MAX_OUTPUT_TOKENS,
            text_format=QuotationSpecAssessmentBatch,
            store=False,
            timeout=60,
        )
        parsed_batch = response.output_parsed
        if parsed_batch is None:
            error = "output_parsed가 비어있음 (구조화 출력 실패)"
    except Exception as exc:  # noqa: BLE001 - 벤치마크용으로 원인 그대로 노출
        error = f"{type(exc).__name__}: {exc}"
    elapsed = time.perf_counter() - started

    assessment = None
    if parsed_batch is not None and parsed_batch.assessments:
        assessment = parsed_batch.assessments[0].model_dump()

    return {
        "engine": "luna",
        "quotation_id": quotation["quotation_id"],
        "elapsed_seconds": elapsed,
        "structured_valid": assessment is not None,
        "error": error,
        "assessment": assessment,
    }


## 5. RunPod Qwen3.5 호출 (`task=spec_eval`)

`prompt_contract.py`가 요구하는 대로 `system_prompt` + `user_prompt`의 SHA-256을
직접 계산해서 같이 보냅니다. RunPod 워커의 구조화 단계는 OpenAI처럼 강제 스키마
파싱 기능이 없으므로, **원하는 JSON 모양을 user_prompt에 직접 설명**해야 합니다.
system_prompt(심사 기준)은 Luna와 동일한 `INSTRUCTIONS`를 그대로 재사용해서
"같은 기준으로 평가시켰는지"를 공정하게 비교합니다.

`/run` 제출 후 `/status/{job_id}`를 폴링하는 방식은 운영 코드
(`backend_logic2/integrations/quotation_extraction/runpod.py`)와 동일합니다.

In [22]:

RUNPOD_SPEC_EVAL_SCHEMA_PROMPT = """
아래 [견적 원문]에는 JSON 형식의 rfq와 quotations(원소 1개)가 들어 있습니다.
INSTRUCTIONS의 판정 기준(출력 분량 제한 포함)에 따라 규격 적합성을 평가하고, 다른 설명 없이
아래 스키마와 정확히 같은 모양의 JSON 객체 하나만 출력하세요. 마크다운 코드블록,
추가 설명, 주석을 절대 붙이지 마세요.

{
  "assessments": [
    {
      "quotation_id": "<quotations[0].quotation_id 값 그대로>",
      "compliant": <true 또는 false>,
      "score": <0~100 사이 숫자>,
      "confidence": <0~1 사이 숫자>,
      "reason": "<1문장 요약>",
      "items": [
        {
          "quotation_item": "<해당 견적 품목명>",
          "rfq_item": "<대응되는 RFQ 품목명>",
          "compliant": <true 또는 false>,
          "score": <0~100 사이 숫자>,
          "reason": "<1~2문장 근거>",
          "missing_or_conflicting_specs": ["<짧은 구 형태의 미확인/상충 항목>"]
        }
      ]
    }
  ]
}
""".strip()


def _prompt_sha256(system_prompt: str, user_prompt: str) -> str:
    return hashlib.sha256(f"{system_prompt}\n{user_prompt}".encode("utf-8")).hexdigest()


def build_spec_eval_worker_input(rfq: dict, quotation: dict) -> dict:
    payload = {
        "rfq": rfq,
        "quotations": [
            {
                "quotation_id": quotation["quotation_id"],
                "supplier_name": quotation["supplier_name"],
                "items": [
                    {
                        "item_code": item["item_code"],
                        "item_name": item["item_name"],
                        "description": item["description"],
                        "raw_description": item["raw_description"],
                        "specifications": item["specifications"],
                    }
                    for item in quotation["items"]
                ],
                "notes": quotation["notes"],
            }
        ],
    }
    document_text = json.dumps(payload, ensure_ascii=False, indent=2)
    system_prompt = INSTRUCTIONS
    user_prompt = RUNPOD_SPEC_EVAL_SCHEMA_PROMPT
    prompt_sha256 = _prompt_sha256(system_prompt, user_prompt)
    return {
        "request_id": f"spec-eval-benchmark-{quotation['quotation_id']}",
        "task": "spec_eval",  # handler.py 패치가 적용돼 있어야 함
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "prompt_version": "spec-eval-benchmark-v1",
        "prompt_sha256": prompt_sha256,
        "pipeline_version": "document-text-v1",
        "documents": [],
        "document_text": document_text,
        "requires_ocr": False,
        "input_mode": "text",
        "max_new_tokens": 900,  # 필요하면 늘리세요 (엔드포인트 MAX_NEW_TOKENS_CAP 확인)
    }


def _runpod_headers() -> dict:
    return {
        "Authorization": f"Bearer {RUNPOD_API_KEY}",
        "Content-Type": "application/json",
    }


def call_runpod_spec_eval(
    rfq: dict,
    quotation: dict,
    *,
    poll_interval_seconds: float = 2.0,
    timeout_seconds: float = 300.0,
) -> dict:
    """단일 견적에 대해 RunPod Qwen3.5로 규격 평가를 1회 호출한다."""

    endpoint_url = f"{RUNPOD_API_BASE_URL}/{RUNPOD_QUOTATION_ENDPOINT_ID}"
    worker_input = build_spec_eval_worker_input(rfq, quotation)

    started = time.perf_counter()
    error = None
    raw_extraction = None
    parsed_assessment = None
    try:
        submit_resp = requests.post(
            f"{endpoint_url}/run",
            headers=_runpod_headers(),
            json={"input": worker_input},
            timeout=(10, 60),
        )
        submit_resp.raise_for_status()
        submit_payload = submit_resp.json()
        status = str(submit_payload.get("status") or "").upper()
        job_id = str(submit_payload.get("id") or "").strip()
        output = None

        if status == "COMPLETED":
            output = submit_payload.get("output")
        elif status in {"CANCELLED", "FAILED", "TIMED_OUT"}:
            error = f"RunPod 작업이 {status} 상태로 즉시 종료됨: {submit_payload}"
        elif not job_id:
            error = f"제출 응답에 작업 ID가 없음: {submit_payload}"
        else:
            deadline = time.monotonic() + timeout_seconds
            while time.monotonic() < deadline:
                time.sleep(poll_interval_seconds)
                status_resp = requests.get(
                    f"{endpoint_url}/status/{job_id}",
                    headers=_runpod_headers(),
                    timeout=(10, 60),
                )
                status_resp.raise_for_status()
                status_payload = status_resp.json()
                status = str(status_payload.get("status") or "").upper()
                if status == "COMPLETED":
                    output = status_payload.get("output")
                    break
                if status in {"CANCELLED", "FAILED", "TIMED_OUT"}:
                    error = f"RunPod 작업이 {status} 상태로 종료됨: {status_payload}"
                    break
            else:
                error = "RunPod 작업이 제한 시간 안에 완료되지 않음 (timeout_seconds를 늘려보세요)"

        worker_metrics = None
        if output is not None and error is None:
            worker_metrics = output.get("metrics")
            if str(output.get("status") or "").lower() != "success":
                error = f"워커가 실패를 반환함: {output}"
            elif output.get("task") != "spec_eval":
                error = (
                    "워커가 task=spec_eval을 인식하지 못함 — handler.py 패치가 "
                    "배포된 버전인지 확인하세요. (재배포 전이면 여기서 계속 실패합니다)"
                )
            else:
                raw_extraction = output.get("extraction")
                try:
                    batch = QuotationSpecAssessmentBatch.model_validate(raw_extraction)
                    if batch.assessments:
                        parsed_assessment = batch.assessments[0].model_dump()
                    else:
                        error = "assessments가 비어있음"
                except Exception as exc:  # noqa: BLE001
                    error = f"모델 출력이 기대한 스키마와 다름: {type(exc).__name__}: {exc}"
    except requests.RequestException as exc:
        error = f"HTTP 요청 실패: {exc}"
    except Exception as exc:  # noqa: BLE001
        error = f"{type(exc).__name__}: {exc}"

    elapsed = time.perf_counter() - started
    # worker_metrics(있으면)에 워커 내부에서 잰 순수 생성 시간(structure_generation_seconds)과
    # 모델 로딩 시간(model_load_seconds)이 따로 들어있습니다. elapsed_seconds(클라이언트가 잰
    # 전체 왕복 시간)에서 이걸 빼보면, 느린 원인이 "모델 생성 자체"인지 "RunPod 대기열/네트워크"
    # 인지 구분할 수 있습니다.
    return {
        "engine": "runpod_qwen35",
        "quotation_id": quotation["quotation_id"],
        "elapsed_seconds": elapsed,
        "structured_valid": parsed_assessment is not None,
        "error": error,
        "assessment": parsed_assessment,
        "raw_extraction": raw_extraction,
        "worker_metrics": worker_metrics,
    }


## 6. 전체 실행: 견적 10건 × 엔진 2개

콜드스타트 영향을 보고 싶으면 이 셀을 한 번 돌리고(워밍업), 다시 한 번 더 돌려서
(웜 상태) 두 결과를 따로 비교하세요. 견적이 10건이라 RunPod 쪽은 웜 상태 기준으로도
총 4~5분 정도 걸릴 수 있습니다(견적 1건당 20~30초대 기준).

In [30]:

results = []
for quotation in QUOTATIONS:
    case_id = quotation["_case_id"]
    print(f"[{case_id}] Luna 호출 중...")
    luna_result = call_luna(RFQ, quotation)
    luna_result["case_id"] = case_id
    results.append(luna_result)

    print(f"[{case_id}] RunPod Qwen3.5 호출 중... (콜드스타트 시 오래 걸릴 수 있음)")
    runpod_result = call_runpod_spec_eval(RFQ, quotation)
    runpod_result["case_id"] = case_id
    results.append(runpod_result)
    if runpod_result.get("worker_metrics"):
        print(f"  worker_metrics: {runpod_result['worker_metrics']}")

print("완료")


[Q1_가상공급사A_완전충족] Luna 호출 중...
[Q1_가상공급사A_완전충족] RunPod Qwen3.5 호출 중... (콜드스타트 시 오래 걸릴 수 있음)
  worker_metrics: {'elapsed_seconds': 21.242842545267195, 'generation_seconds': 21.225275927223265, 'image_views': 0, 'input_mode': 'text', 'model_load_seconds': 19.698938400950283, 'ocr_attempted': False, 'ocr_generation_seconds': 0, 'pages': 0, 'structure_generation_seconds': 21.225275927223265}
[Q2_가상공급사B_상위규격_우량공급사] Luna 호출 중...
[Q2_가상공급사B_상위규격_우량공급사] RunPod Qwen3.5 호출 중... (콜드스타트 시 오래 걸릴 수 있음)
  worker_metrics: {'elapsed_seconds': 20.16985455295071, 'generation_seconds': 20.15103323943913, 'image_views': 0, 'input_mode': 'text', 'model_load_seconds': 19.698938400950283, 'ocr_attempted': False, 'ocr_generation_seconds': 0, 'pages': 0, 'structure_generation_seconds': 20.15103323943913}
[Q3_가상공급사C_외부SCBA_자인] Luna 호출 중...
[Q3_가상공급사C_외부SCBA_자인] RunPod Qwen3.5 호출 중... (콜드스타트 시 오래 걸릴 수 있음)
  worker_metrics: {'elapsed_seconds': 22.324361915234476, 'generation_seconds': 22.308635845780373, 'image_vie

## 7. 결과 비교표

`compliant_match`는 최상위 `compliant` 값이 `ANSWER_KEY`의 정답과 일치하는지입니다.
품목별(`items`) 판정 근거가 타당한지는 표만으로는 알 수 없으니, 아래 8번 셀에서
원문 응답을 직접 읽어보고 정성적으로 확인하세요 — 특히 Q3(상위규격)와 Q6(상충)은
`reason`/`missing_or_conflicting_specs`의 논리가 맞는지가 핵심입니다.

In [31]:

rows = []
for result in results:
    assessment = result.get("assessment") or {}
    predicted_compliant = assessment.get("compliant")
    expected = ANSWER_KEY.get(result["quotation_id"], {})
    expected_compliant = expected.get("expected_compliant")
    compliant_match = (
        predicted_compliant == expected_compliant
        if result["structured_valid"]
        else None
    )
    worker_metrics = result.get("worker_metrics") or {}
    rows.append(
        {
            "case_id": result["case_id"],
            "engine": result["engine"],
            "elapsed_sec": round(result["elapsed_seconds"], 2),
            # RunPod에서만 채워짐: 워커 내부에서 실측한 순수 생성 시간 / 모델 로딩 시간.
            # elapsed_sec - gen_sec - load_sec 차이가 크면 대기열/네트워크가 원인이라는 뜻.
            "gen_sec": round(worker_metrics.get("structure_generation_seconds", 0), 2)
            if worker_metrics.get("structure_generation_seconds") is not None
            else None,
            "load_sec": round(worker_metrics["model_load_seconds"], 2)
            if worker_metrics.get("model_load_seconds") is not None
            else None,
            "structured_valid": result["structured_valid"],
            "predicted_compliant": predicted_compliant,
            "expected_compliant": expected_compliant,
            "compliant_match": compliant_match,
            "score": assessment.get("score"),
            "confidence": assessment.get("confidence"),
            "reason": (assessment.get("reason") or "")[:80],
            "error": result["error"],
        }
    )

if pd is not None:
    df = pd.DataFrame(rows)
    display(df)
else:
    for row in rows:
        print(row)


,case_id,engine,elapsed_sec,gen_sec,load_sec,structured_valid,predicted_compliant,expected_compliant,compliant_match,score,confidence,reason,error
0,Q1_가상공급사A_완전충족,luna,2.99,NaN,NaN,True,True,True,True,100.0,0.99,제시된 품목은 RFQ의 모든 필수 기술 규격을 충족하거나 상회합니다.,None
1,Q1_가상공급사A_완전충족,runpod_qwen35,22.02,21.23,19.7,True,True,True,True,100.0,1.00,"RFQ의 모든 필수 규격이 견적품목의 사양을 완전히 충족하거나, 허용오차 범위 내에...",None
2,Q2_가상공급사B_상위규격_우량공급사,luna,6.52,NaN,NaN,True,False,True,False,92.0,0.95,대부분의 필수 규격을 충족하지만 봉제부의 보호복 본체 동등 투과 저항과 보관 수명의...,None
3,Q2_가상공급사B_상위규격_우량공급사,runpod_qwen35,21.89,20.15,19.7,True,True,True,True,100.0,1.00,RFQ의 모든 필수 규격이 견적품목의 사양을 포함하거나 초과하여 충족합니다.,None
4,Q3_가상공급사C_외부SCBA_자인,luna,5.08,NaN,NaN,True,False,False,True,64.0,0.99,주요 재질·투과저항·인증 등은 충족하지만 보호복 형식과 SCBA 위치가 RFQ와 상...,None
5,Q3_가상공급사C_외부SCBA_자인,runpod_qwen35,24.64,22.31,19.7,True,False,False,True,0.0,1.00,RFQ 요구사항인 내부 SCBA 착용 방식과 견적의 외부 SCBA 방식이 상충하여 ...,None
6,Q4_가상공급사D_전면미달_자인,luna,6.40,NaN,NaN,True,False,False,True,15.0,0.99,필수 규격 다수가 미충족되거나 확인되지 않아 RFQ 기술 규격에 부적합합니다.,None
7,Q4_가상공급사D_전면미달_자인,runpod_qwen35,46.28,43.53,19.7,True,False,False,True,0.0,1.00,"필수 인증(KCs, EN 464), 보호복 두께, 투과 저항 시간, 안면창 시야각,...",None
8,Q5_가상공급사E_KCs인증_미확정,luna,4.81,NaN,NaN,True,False,False,True,69.2,0.99,대부분의 성능 규격은 일치하지만 KCs 인증 보장과 일부 필수 세부 규격이 확인되지...,None
9,Q5_가상공급사E_KCs인증_미확정,runpod_qwen35,21.48,19.71,19.7,True,False,False,True,85.0,0.95,KC 인증 발급 불확실성으로 인해 필수 인증 규격 미충족 및 공기호흡기 용량 하한선...,None


## 8. 요약 지표

엔진별로: 평균 지연시간, 구조적으로 유효한 JSON을 낸 비율, 정답(compliant) 일치율.
견적 10건짜리 소규모 테스트라 통계적으로 확정적인 결론을 내리기보단, "RunPod 쪽이
아직 스키마를 잘 못 지킨다" / "단위 환산 트랩(Q6, Q9)에서 유독 틀린다" / "자인형
부적합(Q3, Q4, Q8)은 둘 다 잘 잡는다" 같은 패턴을 찾는 용도로 보세요.

In [32]:

def summarize(engine: str) -> dict:
    engine_rows = [r for r in rows if r["engine"] == engine]
    n = len(engine_rows)
    valid_rows = [r for r in engine_rows if r["structured_valid"]]
    matched_rows = [r for r in valid_rows if r["compliant_match"] is True]
    avg_latency = sum(r["elapsed_sec"] for r in engine_rows) / n if n else float("nan")
    return {
        "engine": engine,
        "cases": n,
        "avg_latency_sec": round(avg_latency, 2),
        "structured_valid_rate": f"{len(valid_rows)}/{n}",
        "compliant_accuracy": (
            f"{len(matched_rows)}/{len(valid_rows)}" if valid_rows else "N/A"
        ),
    }


summary_rows = [summarize("luna"), summarize("runpod_qwen35")]
if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    for row in summary_rows:
        print(row)


,engine,cases,avg_latency_sec,structured_valid_rate,compliant_accuracy
0,luna,10,5.32,10/10,7/10
1,runpod_qwen35,10,27.14,10/10,9/10


## 9. 원문 응답 직접 확인 (정성 평가용)

케이스 ID를 바꿔가며 두 엔진의 `reason`/`items`/`missing_or_conflicting_specs`를
직접 읽어보세요. 특히 Q3(상위규격 인정 여부)과 Q6(구조화 필드-원문 상충 감지 여부)은
숫자 점수만으로는 판단하기 어렵습니다.

In [35]:

def show_case(case_id: str):
    for result in results:
        if result["case_id"] != case_id:
            continue
        print(f"--- {result['engine']} ({case_id}) ---")
        print(f"elapsed: {result['elapsed_seconds']:.2f}s | structured_valid: {result['structured_valid']}")
        if result["error"]:
            print(f"error: {result['error']}")
        print(json.dumps(result.get("assessment"), ensure_ascii=False, indent=2))
        print()


# 이번 세트에서 가장 어려운 케이스(영문 + 5종 단위환산)부터 확인해보세요.
show_case("Q4_가상공급사I_영문_다중단위환산")


In [37]:
for r in results:
    if r["case_id"] == "Q4_가상공급사D_전면미달_자인" and r["engine"] == "runpod_qwen35":
        print(json.dumps(r["assessment"], ensure_ascii=False, indent=2))

{
  "quotation_id": "SUP-QTN-SPEC-004",
  "compliant": false,
  "score": 0.0,
  "confidence": 1.0,
  "reason": "필수 인증(KCs, EN 464), 보호복 두께, 투과 저항 시간, 안면창 시야각, 장갑/안전화 성능, 사용 온도, 보관 수명, 사이즈 등 13 개 필수 규격 중 대부분 미달 또는 상충하여 적합하지 않습니다.",
  "items": [
    {
      "quotation_item": "SafeFlex 밀폐형 화학보호복",
      "rfq_item": "화학물질용 완전밀폐형 전신 보호복",
      "compliant": false,
      "score": 0.0,
      "reason": "필수 인증(KCs, EN 464), 보호복 두께 (0.45mm<0.50mm), 투과 저항 시간 (황산 180<480, 암모니아 120<240, 염소 자료없음), 안면창 시야각 (75<80), 장갑 (고정형), 안전화 (100J<200J), 사용 온도 (-10~-30, +50<+60), 보관 수명 (3<5), 사이즈 (M 없음) 등 13 개 필수 규격이 모두 불충족됩니다.",
      "missing_or_conflicting_specs": [
        "KCs 안전인증 미포함",
        "EN 464 기밀 시험 미포함",
        "재질 두께 0.45mm<0.50mm 요구",
        "황산 투과 저항 180분<480분 요구",
        "암모니아 투과 저항 120분<240분 요구",
        "염소가스 투과 저항 자료 없음",
        "시야각 75도<80도 요구",
        "장갑 고정형 (교체형 미구비)",
        "안전화 토캡 100J<200J 요구",
        "사용 온도 -10℃< -30℃ 요구",
        "사용 온도 +50℃< +60℃ 요구",
        "보관 수명 3년<5년 

In [44]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def _run_batch(fn, quotations, label):
    """quotations 전체를 동시에(concurrent) 호출해서, 순차 실행 대비
    wall-clock이 얼마나 줄어드는지 (=RunPod가 실제로 병렬 스케일링되는지) 확인용."""
    started = time.perf_counter()
    out = {}
    with ThreadPoolExecutor(max_workers=len(quotations)) as executor:
        futures = {
            executor.submit(fn, RFQ, q): q["_case_id"] for q in quotations
        }
        for future in as_completed(futures):
            case_id = futures[future]
            try:
                result = future.result()
                result["case_id"] = case_id
                out[case_id] = result
                elapsed = time.perf_counter() - started
                print(f"[{label}][{case_id}] 완료 (배치 시작 후 {elapsed:.1f}s)")
            except Exception as e:
                out[case_id] = {"case_id": case_id, "error": str(e)}
                print(f"[{label}][{case_id}] 실패: {e}")
    wall_clock = time.perf_counter() - started
    print(f"\n[{label}] {len(quotations)}건 동시 실행 wall-clock: {wall_clock:.1f}초\n")
    return out, wall_clock


# RunPod 병렬 스케일링 확인 (원래 궁금해하셨던 "병렬처리로 커버되나" 질문의 핵심)
runpod_batch_results, runpod_wall_clock = _run_batch(
    call_runpod_spec_eval, QUOTATIONS, "runpod_qwen35"
)

# 참고 비교용 (Luna는 별개 엔진이라 따로 실행 — 섞으면 어느 쪽 지연인지 구분 안 됨)
luna_batch_results, luna_wall_clock = _run_batch(
    call_luna, QUOTATIONS, "luna"
)

print(f"RunPod 10건 동시: {runpod_wall_clock:.1f}s")
print(f"Luna   10건 동시: {luna_wall_clock:.1f}s")
print("※ 순차 실행 대비 비교하려면 아까 QUOTATIONS[:2]/전체 순차 실행 때의 case별 elapsed_sec 합과 비교하세요.")

[runpod_qwen35][Q3_가상공급사C_외부SCBA_자인] 완료 (배치 시작 후 20.3s)
[runpod_qwen35][Q6_가상공급사F_구조화누락_단위환산] 완료 (배치 시작 후 29.9s)
[runpod_qwen35][Q4_가상공급사D_전면미달_자인] 완료 (배치 시작 후 49.4s)
[runpod_qwen35][Q2_가상공급사B_상위규격_우량공급사] 완료 (배치 시작 후 51.5s)
[runpod_qwen35][Q1_가상공급사A_완전충족] 완료 (배치 시작 후 66.0s)
[runpod_qwen35][Q7_가상공급사G_스펙모호_사이즈누락] 완료 (배치 시작 후 80.4s)
[runpod_qwen35][Q9_가상공급사I_영문_다중단위환산] 완료 (배치 시작 후 85.2s)
[runpod_qwen35][Q5_가상공급사E_KCs인증_미확정] 완료 (배치 시작 후 101.8s)
[runpod_qwen35][Q10_가상공급사J_구조화vs원문_대체모델리스크] 완료 (배치 시작 후 103.8s)
[runpod_qwen35][Q8_가상공급사H_다른표준_자인] 완료 (배치 시작 후 132.5s)

[runpod_qwen35] 10건 동시 실행 wall-clock: 132.5초

[luna][Q1_가상공급사A_완전충족] 완료 (배치 시작 후 3.2s)
[luna][Q2_가상공급사B_상위규격_우량공급사] 완료 (배치 시작 후 4.8s)
[luna][Q9_가상공급사I_영문_다중단위환산] 완료 (배치 시작 후 4.9s)
[luna][Q4_가상공급사D_전면미달_자인] 완료 (배치 시작 후 5.1s)
[luna][Q7_가상공급사G_스펙모호_사이즈누락] 완료 (배치 시작 후 5.5s)
[luna][Q6_가상공급사F_구조화누락_단위환산] 완료 (배치 시작 후 6.1s)
[luna][Q3_가상공급사C_외부SCBA_자인] 완료 (배치 시작 후 6.3s)
[luna][Q8_가상공급사H_다른표준_자인] 완료 (배치 시작 후 6.9s)
[luna][Q10_가상공급사J_구조화vs원문_대

In [46]:
batch_rows = []
for engine_label, batch_results in (("luna", luna_batch_results), ("runpod_qwen35", runpod_batch_results)):
    for case_id, result in batch_results.items():
        # ThreadPoolExecutor의 future.result()에서 예외로 잡힌 경우(call_* 함수 자체가
        # 반환하는 정상 실패 형태와 모양이 다름) 맞춰주기
        if "error" in result and "assessment" not in result:
            result = {
                "engine": engine_label,
                "quotation_id": None,
                "elapsed_seconds": None,
                "structured_valid": False,
                "error": result["error"],
                "assessment": None,
                "worker_metrics": None,
                "case_id": case_id,
            }

        assessment = result.get("assessment") or {}
        predicted_compliant = assessment.get("compliant")
        expected = ANSWER_KEY.get(result.get("quotation_id"), {})
        expected_compliant = expected.get("expected_compliant")
        compliant_match = (
            predicted_compliant == expected_compliant
            if result.get("structured_valid")
            else None
        )
        worker_metrics = result.get("worker_metrics") or {}
        batch_rows.append(
            {
                "case_id": result.get("case_id", case_id),
                "engine": result.get("engine", engine_label),
                "elapsed_sec": round(result["elapsed_seconds"], 2)
                if result.get("elapsed_seconds") is not None else None,
                "gen_sec": round(worker_metrics.get("structure_generation_seconds", 0), 2)
                if worker_metrics.get("structure_generation_seconds") is not None else None,
                "load_sec": round(worker_metrics["model_load_seconds"], 2)
                if worker_metrics.get("model_load_seconds") is not None else None,
                "structured_valid": result.get("structured_valid", False),
                "predicted_compliant": predicted_compliant,
                "expected_compliant": expected_compliant,
                "compliant_match": compliant_match,
                "score": assessment.get("score"),
                "confidence": assessment.get("confidence"),
                "reason": (assessment.get("reason") or "")[:80],
                "error": result.get("error"),
            }
        )

if pd is not None:
    batch_df = pd.DataFrame(batch_rows)
    display(batch_df)
else:
    for row in batch_rows:
        print(row)


def summarize_batch(engine: str) -> dict:
    engine_rows = [r for r in batch_rows if r["engine"] == engine]
    n = len(engine_rows)
    valid_rows = [r for r in engine_rows if r["structured_valid"]]
    matched_rows = [r for r in valid_rows if r["compliant_match"] is True]
    timed_rows = [r for r in engine_rows if r["elapsed_sec"] is not None]
    avg_latency = (
        sum(r["elapsed_sec"] for r in timed_rows) / len(timed_rows)
        if timed_rows else float("nan")
    )
    return {
        "engine": engine,
        "cases": n,
        "avg_latency_sec": round(avg_latency, 2),
        "structured_valid_rate": f"{len(valid_rows)}/{n}",
        "compliant_accuracy": f"{len(matched_rows)}/{len(valid_rows)}" if valid_rows else "N/A",
    }


batch_summary_rows = [summarize_batch("luna"), summarize_batch("runpod_qwen35")]
if pd is not None:
    display(pd.DataFrame(batch_summary_rows))
else:
    for row in batch_summary_rows:
        print(row)

print(f"\nwall-clock 비교 — runpod: {runpod_wall_clock:.1f}s, luna: {luna_wall_clock:.1f}s")
print("avg_latency_sec는 '건당 평균'이고 wall-clock은 '10건 동시에 돌렸을 때 전체가 끝난 실제 시간'입니다.")
print("wall-clock ≈ avg_latency_sec → 병렬 스케일링 잘 됨 / wall-clock ≈ avg_latency_sec × 10 → 사실상 순차 처리(스케일링 안 됨)")

,case_id,engine,elapsed_sec,gen_sec,load_sec,structured_valid,predicted_compliant,expected_compliant,compliant_match,score,confidence,reason,error
0,Q1_가상공급사A_완전충족,luna,3.19,NaN,NaN,True,True,True,True,100.0,0.99,"제시된 보호복 형식, 인증, 재질, 투과 저항, 봉제부, 부속품, 시험·사용 조건 ...",NaN
1,Q2_가상공급사B_상위규격_우량공급사,luna,4.79,NaN,NaN,True,False,True,False,92.3,0.98,대부분의 필수 규격을 충족하지만 봉제부가 보호복 본체와 동등한 화학물질 투과 저항을...,NaN
2,Q9_가상공급사I_영문_다중단위환산,luna,4.80,NaN,NaN,True,False,True,False,92.3,0.98,대부분의 필수 규격은 충족하지만 안전화의 보호복 일체형 여부가 확인되지 않아 전체 ...,NaN
3,Q4_가상공급사D_전면미달_자인,luna,5.06,NaN,NaN,True,False,False,True,7.0,0.99,필수 규격 다수가 미충족하거나 확인되지 않아 부적합입니다.,NaN
4,Q7_가상공급사G_스펙모호_사이즈누락,luna,5.43,NaN,NaN,True,False,False,True,5.0,0.99,"완전 밀폐형이라는 일부 요건과 L/XL 사이즈만 확인되며, 다수의 필수 인증·성능·...",NaN
5,Q6_가상공급사F_구조화누락_단위환산,luna,6.05,NaN,NaN,True,False,True,False,85.7,0.97,대부분의 필수 규격은 충족하지만 봉제부 투과 저항 동등성과 일련번호별 EN 464 ...,NaN
6,Q3_가상공급사C_외부SCBA_자인,luna,6.29,NaN,NaN,True,False,False,True,70.0,0.99,Type 1a-ET 및 내부 SCBA가 요구되나 Type 1b-ET 외부 SCBA를...,NaN
7,Q8_가상공급사H_다른표준_자인,luna,6.82,NaN,NaN,True,False,False,True,7.0,0.99,요구된 EN 943-1 Type 1a-ET 완전밀폐형 보호복과 상충하는 EN 146...,NaN
8,Q10_가상공급사J_구조화vs원문_대체모델리스크,luna,6.92,NaN,NaN,True,False,False,True,85.0,0.97,대부분의 필수 규격은 충족하지만 안전화의 EN ISO 20345·일체형 여부와 봉제...,NaN
9,Q5_가상공급사E_KCs인증_미확정,luna,8.54,NaN,NaN,True,False,False,True,85.0,0.99,대부분의 보호 성능과 구조 사양은 충족하지만 필수 KCs 인증과 일부 세부 성능이 ...,NaN


,engine,cases,avg_latency_sec,structured_valid_rate,compliant_accuracy
0,luna,10,5.79,10/10,7/10
1,runpod_qwen35,10,72.07,9/10,8/9



wall-clock 비교 — runpod: 132.5s, luna: 8.6s
avg_latency_sec는 '건당 평균'이고 wall-clock은 '10건 동시에 돌렸을 때 전체가 끝난 실제 시간'입니다.
wall-clock ≈ avg_latency_sec → 병렬 스케일링 잘 됨 / wall-clock ≈ avg_latency_sec × 10 → 사실상 순차 처리(스케일링 안 됨)


In [47]:
for row in batch_rows:
    if row["engine"] == "runpod_qwen35" and not row["structured_valid"]:
        print(row["case_id"])
        print(row["error"])

Q5_가상공급사E_KCs인증_미확정
모델 출력이 기대한 스키마와 다름: ValidationError: 1 validation error for QuotationSpecAssessmentBatch
assessments.0.confidence
  Field required [type=missing, input_value={'compliant': False, 'ite... 미달.', 'score': 85}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
